# 45 — Requirement Classification
**Goal:** Classify JD requirements as must-have, nice-to-have, or preferred using zero-shot.

The final JD-side chapter labels each requirement line with its *weight class*: **must-have** (a screening gate), **nice-to-have** (a ranking bonus), or **preferred** (a soft plus). Two approaches are compared head-to-head: **zero-shot classification**, which uses a pretrained natural-language-inference model to judge each line against the three labels with no training data, and **rule-based classification**, which scans for trigger words like "preferred" or "must".

**Why it matters for resumes / ATS:** Ch. 46 cannot score a resume against a JD until each requirement knows its weight. A missing must-have should cap the score; a missing nice-to-have should barely dent it. Misclassifying "PhD preferred" as must-have would reject strong candidates who lack the degree. Zero-shot offers accuracy at the cost of a large model; rules offer instant, offline, explainable verdicts — the trade-off this chapter makes visible.

## 1. Zero-Shot Classification

Zero-shot classification repurposes an **NLI (natural language inference)** model: `facebook/bart-large-mnli` was trained to decide whether one sentence *entails* another. To classify "PhD in Computer Science preferred", the pipeline treats each label ("must-have", "nice-to-have", "preferred") as a hypothesis and scores how strongly the requirement entails it — no task-specific fine-tuning needed.

**What the code does:**
- `pipeline("zero-shot-classification", model="facebook/bart-large-mnli")` builds the classifier (a multi-hundred-MB download on first use).
- Each requirement is classified against the three labels; `result['labels'][0]` and `result['scores'][0]` give the top label and its confidence.
- A `try/except` wraps everything: if transformers or the model is unavailable, it prints "Transformers not available" and falls back to a keyword rule (words like "preferred"/"plus"/"nice"/"bonus" → nice-to-have, else must-have).

**Expected:** with the model available, the classifier reads meaning rather than keywords: "5+ years experience in Python" → `must-have`; "Published research papers a plus" → `nice-to-have`; and an implicit one like "Ability to work in fast-paced environment" gets judged on content, not trigger words. The price is the download and per-line latency; in environments without the model (like a fresh offline install), the except-branch rule fallback is what actually runs.

In [ ]:
from transformers import pipeline
requirements = [
    "5+ years experience in Python",
    "Strong communication skills",
    "PhD in Computer Science preferred",
    "Experience with AWS or GCP",
    "Published research papers a plus",
    "Ability to work in fast-paced environment",
]

try:
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
    labels = ["must-have", "nice-to-have", "preferred"]
    
    for req in requirements:
        result = classifier(req, labels)
        print(f"  '{req[:40]:40s}' -> {result['labels'][0]:15s} ({result['scores'][0]:.2f})")
except:
    print("Transformers not available. Rule-based fallback:")
    for req in requirements:
        if any(w in req.lower() for w in ["preferred", "plus", "nice", "bonus"]):
            print(f"  '{req[:40]:40s}' -> nice-to-have (rule)")
        else:
            print(f"  '{req[:40]:40s}' -> must-have (rule)")

## 2. Rule-Based Classification

The rule-based alternative is a **keyword cascade**: check the soft signals first, then the hard signals, then soft-skill phrases — first hit wins, and anything unmatched defaults to must-have. It is deterministic, instant, and needs no model.

**What the code does:**
- `classify_requirement()` lowercases the text and runs three `any(w in text_lower for w in [...])` checks in priority order: soft words ("preferred", "plus", "nice", "bonus", "desired") → `nice-to-have`; hard words ("must", "required", "essential", "minimum") → `must-have`; phrase signals ("ability to", "strong") → `soft-skill`.
- Unmatched text returns `must-have` as the conservative default — better to over-require than under-require in screening.
- The loop classifies the same six requirements as the zero-shot cell, so the two approaches can be compared line by line.

**Expected:** verified on the shared list: "5+ years experience in Python" and "Experience with AWS or GCP" → `must-have`; "PhD in Computer Science preferred" and "Published research papers a plus" → `nice-to-have`; "Strong communication skills" and "Ability to work in fast-paced environment" → `soft-skill`. The cascade is transparent — every verdict traces to one trigger word — which is exactly its advantage and its limit: rephrase "preferred" as "would be great" and the rule misses it, while the zero-shot model would not.

In [ ]:
def classify_requirement(text):
    """Classify requirement by keyword rules."""
    text_lower = text.lower()
    if any(w in text_lower for w in ["preferred", "plus", "nice", "bonus", "desired"]):
        return "nice-to-have"
    if any(w in text_lower for w in ["must", "required", "essential", "minimum"]):
        return "must-have"
    if any(w in text_lower for w in ["ability to", "strong"]):
        return "soft-skill"
    return "must-have"  # default

for req in requirements:
    print(f"  {classify_requirement(req):15s} | {req}")

## Summary: Zero-shot is more accurate. Rule-based is faster and always available.

**Requirement weighting is the last input Ch. 46 needs — and the choice between zero-shot and rules is a real engineering trade-off, not a stylistic one.**

Zero-shot classification reads semantics and generalizes to unseen phrasing, but needs a large pretrained model and network access; rules are instant, offline, and explainable, but only as good as their trigger-word lists. The verified rule output on the six sample requirements (`must-have` for the hard gates, `nice-to-have` for "preferred"/"plus", `soft-skill` for the soft phrases) shows the cascade is a solid default, with the model as the upgrade path. This completes the JD profile: sections (Ch. 40), skills (Ch. 41), responsibilities (Ch. 42), qualifications (Ch. 43), ranked keywords (Ch. 44), and weighted requirements — ready for Ch. 46, Resume vs JD Matching.